# A04 · 08 · Synthesis & hypothesis verdict
Programmatic decision tree against the A06 predictions, plus the write-up scaffold (see `docs/A04_results_discussion_template.md`).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))   # a04_common.py next to notebooks
import a04_common as ac
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
print('project root:', ac.PROJ)


project root: /storage/users/job37yv/Projects/DECIDE_A04_CdiffOrganoid_Butyrate


In [2]:
id2sym, sym2id = ac.load_symbol_map()
def load(name): return pd.read_csv(ac.TAB/f'DE_{name}.csv', index_col=0)
def lfc(name, gene, key='log2FoldChange'):
    r=load(name); gid=sym2id.get(gene)
    return (r.loc[gid,key] if gid in r.index else np.nan)
def padj(name, gene): return lfc(name, gene, 'padj')

In [3]:
chap = ['HSPA5','CALR','WFS1']; sens = ['XBP1','ATF6','ATF4','EIF2AK3']; chemo=['CXCL1','CXCL2','CXCL3','CXCL8']
# 1) chaperones up with butyrate (Cd arm)?
chap_up = np.nanmean([lfc('Butyrate_10v0_Cd',g) for g in chap])
# 2) sensors flat?
sens_abs = np.nanmean([abs(lfc('Butyrate_10v0_Cd',g)) for g in sens])
# 3) chemokines fall with butyrate in Cd?
chemo_d  = np.nanmean([lfc('Butyrate_10v0_Cd',g) for g in chemo])
# 4) Cd-vs-Mock response shrinks at 10mM?
shrink = np.nanmean([abs(lfc('Cd_vs_Mock_10mM',g)) for g in chemo]) < np.nanmean([abs(lfc('Cd_vs_Mock_0mM',g)) for g in chemo])
print(f'chaperone mean LFC (but10v0,Cd): {chap_up:+.2f}  (expect >0)')
print(f'sensor mean |LFC|             : {sens_abs:.2f}  (expect ~0, <0.3)')
print(f'chemokine mean LFC (but10v0)  : {chemo_d:+.2f}  (expect <0)')
print(f'infection chemokine response shrinks at 10mM: {shrink}')

chaperone mean LFC (but10v0,Cd): -0.63  (expect >0)
sensor mean |LFC|             : 0.40  (expect ~0, <0.3)
chemokine mean LFC (but10v0)  : -3.03  (expect <0)
infection chemokine response shrinks at 10mM: True


In [4]:
verdict = (chap_up>0.3) and (sens_abs<0.4) and (chemo_d<0) and shrink
print('\n=== VERDICT ===')
if verdict:
    print('Butyrate reproduces the A06 non-canonical-UPR + chemokine-attenuation signature.')
elif chap_up>0.3 and sens_abs>=0.4:
    print('Chaperones up but sensors also move -> (partly) CANONICAL UPR here; revise A06 framing for organoid.')
elif not (chap_up>0.3):
    print('No butyrate chaperone effect -> butyrate is NOT the A06 driver; favors AR/commensal-specific mechanism.')
else:
    print('Mixed -> inspect panels in notebook 04 and GSEA in 07 before concluding.')


=== VERDICT ===
No butyrate chaperone effect -> butyrate is NOT the A06 driver; favors AR/commensal-specific mechanism.


## Write-up scaffold
Fill the per-contrast numbers into `docs/A04_results_discussion_template.md`. Compare against A06 LFCs (HSPA5/CALR/WFS1 induced; XBP1/ATF6/ATF4 flat; CXCL1/2/3/8 attenuated 70–100% in co-infection). State caveats: single donor (within-donor only), first-look DE, host-only (no bacterial readout).